In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Chargement du fichier propre
chemin = "../data/propre/rendez_vous_medirdv_propre.csv"
df = pd.read_csv(chemin, encoding="utf-8")

# Conversion des dates
df["date_rdv"] = pd.to_datetime(df["date_rdv"], errors="coerce")
df["date_prise"] = pd.to_datetime(df["date_prise"], errors="coerce")
df["date_naissance"] = pd.to_datetime(df["date_naissance"], errors="coerce")

# Colonnes utiles
df["jour_semaine"] = df["date_rdv"].dt.day_name()
df["mois"] = df["date_rdv"].dt.to_period("M").astype(str)
df["heure"] = df["heure_rdv"].str[:2].astype(float)

# Âge au moment du RDV
df["age"] = (df["date_rdv"] - df["date_naissance"]).dt.days / 365.25

# Délai de prise
df["delai_jours"] = (df["date_rdv"] - df["date_prise"]).dt.days

print("Dimensions :", df.shape)
print("Période :", df["date_rdv"].min(), "→", df["date_rdv"].max())
df.head()

Dimensions : (2015, 25)
Période : 2025-10-01 00:00:00 → 2026-12-31 00:00:00


,id_rdv,date_prise,date_rdv,heure_rdv,duree_min,specialite,medecin_id,medecin_nom,patient_id,patient_nom,...,canal,confirme,statut,motif_annulation,agent_saisie,jour_semaine,mois,heure,age,delai_jours
0,RDV01965,2026-12-12,2026-12-24,16:00,30.0,Gynécologie-obstétrique,MED014,Dr Aliou Ndiaye,PAT00480,Gomis,...,Guichet,Non,Planifié,NaN,AGT07,Thursday,2026-12,16.0,35.238877,12
1,RDV00432,2025-12-30,2026-01-03,10:20,20.0,Ophtalmologie,MED020,Dr Adama Badji,PAT00333,Camara,...,Guichet,Non,Honoré,NaN,AGT03,Saturday,2026-01,10.0,12.199863,4
2,RDV00708,2026-03-03,2026-03-09,17:15,15.0,Médecine générale,MED003,Dr Seynabou Dia,PAT00975,Badji,...,Non renseigné,Non,Honoré,NaN,AGT01,Monday,2026-03,17.0,73.489391,6
3,RDV00331,2025-12-15,2025-12-15,09:00,30.0,Cardiologie,MED017,Dr Oumou Seck,PAT00593,Mbaye,...,Téléphone,Oui,Honoré,NaN,AGT07,Monday,2025-12,9.0,4.479124,0
4,RDV00340,2025-12-15,2025-12-16,10:20,20.0,Pédiatrie,MED009,Dr Youssou Fall,PAT00573,Sow,...,Téléphone,Oui,Honoré,NaN,AGT04,Tuesday,2025-12,10.0,14.266940,1


In [2]:
# --- EF-20 : Indicateurs clés ---
print("--- STATUTS ---")
print(df["statut"].value_counts())

print("\n--- TAUX D'ABSENCE GLOBAL ---")
total_echus = df[df["statut"].isin(["Honoré", "Absent"])].shape[0]
absents = df[df["statut"] == "Absent"].shape[0]
taux_absence = absents / total_echus * 100
print(f"RDV échus (Honoré + Absent) : {total_echus}")
print(f"Absents : {absents}")
print(f"Taux d'absence : {taux_absence:.2f}%")
print("Objectif cahier des charges : 14% → 11%")

print("\n--- TAUX D'OCCUPATION ---")
rdv_actifs = df[~df["statut"].isin(["Annulé"])].shape[0]
print(f"RDV actifs (hors annulés) : {rdv_actifs}")

print("\n--- SPÉCIALITÉS ---")
print(df["specialite"].value_counts())

--- STATUTS ---
statut
Honoré      1267
Planifié     231
Absent       206
Confirmé     160
Annulé       151
Name: count, dtype: int64

--- TAUX D'ABSENCE GLOBAL ---
RDV échus (Honoré + Absent) : 1473
Absents : 206
Taux d'absence : 13.99%
Objectif cahier des charges : 14% → 11%

--- TAUX D'OCCUPATION ---
RDV actifs (hors annulés) : 1864

--- SPÉCIALITÉS ---
specialite
Médecine générale          552
Pédiatrie                  306
Gynécologie-obstétrique    301
Cardiologie                186
Endocrinologie             166
Ophtalmologie              154
Dermatologie               146
ORL                        108
Kinésithérapie              96
Name: count, dtype: int64


In [3]:
# --- EF-21 : Répartition par jour de semaine ---
jours_ordre = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
df_jour = df[df["jour_semaine"].isin(jours_ordre)].copy()

repartition_jour = df_jour.groupby("jour_semaine").size().reindex(jours_ordre)
print("--- RDV par jour de semaine ---")
print(repartition_jour)

fig = px.bar(
    x=repartition_jour.index,
    y=repartition_jour.values,
    labels={"x": "Jour", "y": "Nombre de RDV"},
    title="Répartition des RDV par jour de semaine"
)
fig.show()

--- RDV par jour de semaine ---
jour_semaine
Monday       469
Tuesday      258
Wednesday    231
Thursday     305
Friday       326
Saturday     426
dtype: int64


In [4]:
# --- EF-21 : Carte de chaleur jour × heure ---
df_jour["heure_int"] = df_jour["heure_rdv"].str[:2].astype(int)

pivot = df_jour.pivot_table(
    index="jour_semaine",
    columns="heure_int",
    values="id_rdv",
    aggfunc="count",
    fill_value=0
).reindex(jours_ordre)

fig = px.imshow(
    pivot,
    labels=dict(x="Heure", y="Jour", color="Nb RDV"),
    title="Carte de chaleur : RDV par jour et par heure",
    color_continuous_scale="YlOrRd",
    aspect="auto"
)
fig.show()

In [5]:
# --- EF-22 : Évolution mensuelle du volume et du taux d'absence ---
df_mois = df.copy()
df_mois["mois"] = df_mois["date_rdv"].dt.to_period("M").astype(str)

# Volume total par mois
volume = df_mois.groupby("mois").size()

# Taux d'absence par mois
df_echus = df_mois[df_mois["statut"].isin(["Honoré", "Absent"])]
taux = df_echus.groupby("mois").apply(
    lambda x: (x["statut"] == "Absent").mean() * 100
)

fig = go.Figure()
fig.add_trace(go.Bar(x=volume.index, y=volume.values, name="Volume RDV"))
fig.add_trace(go.Scatter(x=taux.index, y=taux.values, name="Taux d'absence (%)", yaxis="y2"))
fig.update_layout(
    title="Évolution mensuelle : volume et taux d'absence",
    yaxis=dict(title="Nombre de RDV"),
    yaxis2=dict(title="Taux d'absence (%)", overlaying="y", side="right"),
    hovermode="x unified"
)
fig.show()